[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/QuantLet/NEURAL_BIZ/blob/main/Bishop_Ch_01/NB_bishop_ch01_figures.ipynb)

# Bishop Chapter 1 — Key Figures

This notebook programmatically generates all key figures from Bishop *Deep Learning: Foundations and Concepts*, Chapter 1.

Figures are saved as both PDF and PNG (300 dpi) with transparent backgrounds.

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
import numpy as np
import os

# ── Global styling ──────────────────────────────────────────────────
matplotlib.rcParams['figure.facecolor']    = 'none'
matplotlib.rcParams['axes.facecolor']      = 'none'
matplotlib.rcParams['savefig.facecolor']   = 'none'
matplotlib.rcParams['savefig.transparent'] = True
matplotlib.rcParams['axes.grid']           = False
matplotlib.rcParams['legend.framealpha']   = 0.0
matplotlib.rcParams['font.size']           = 11

COLORS = {
    'blue':   '#1A3A6E',
    'red':    '#CD0000',
    'green':  '#2E7D32',
    'amber':  '#B5853F',
    'purple': '#8E44AD',
}

CHART_DIR = os.path.join(os.path.dirname(os.path.abspath('.')), 'charts')
# fallback for Colab
if not os.path.exists(os.path.dirname(CHART_DIR)):
    CHART_DIR = './charts'
os.makedirs(CHART_DIR, exist_ok=True)
print(f'Charts will be saved to: {CHART_DIR}')


def save_fig(fig, name, chart_dir=None):
    """Save figure as PDF + PNG with transparent background."""
    if chart_dir is None:
        chart_dir = CHART_DIR
    for ax in fig.get_axes():
        ax.grid(False)
        legend = ax.get_legend()
        if legend:
            legend.set_bbox_to_anchor((0.5, -0.15))
            legend.set_loc('upper center')
            legend._ncols = min(len(legend.get_texts()), 4)
    fig.savefig(f'{chart_dir}/{name}.pdf', bbox_inches='tight', transparent=True)
    fig.savefig(f'{chart_dir}/{name}.png', bbox_inches='tight', transparent=True, dpi=300)
    print(f'Saved: {name}.pdf / .png')

## Shared Data — Polynomial Fitting

In [ ]:
np.random.seed(42)
N = 10
x = np.linspace(0, 1, N)
t = np.sin(2 * np.pi * x) + np.random.randn(N) * 0.3
x_plot = np.linspace(0, 1, 200)

# Generate a larger test set for RMSE evaluation
np.random.seed(99)
N_test = 100
x_test = np.random.rand(N_test)
t_test = np.sin(2 * np.pi * x_test) + np.random.randn(N_test) * 0.3

## Figure 1.4 — Polynomial Data

N = 10 data points from sin(2*pi*x) + Gaussian noise.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(x_plot, np.sin(2 * np.pi * x_plot), color=COLORS['green'], lw=2, label=r'$\sin(2\pi x)$')
ax.scatter(x, t, color=COLORS['blue'], s=50, zorder=5, edgecolors='white', linewidths=0.5, label='Training data')
ax.set_xlabel('$x$')
ax.set_ylabel('$t$')
ax.set_xlim(-0.05, 1.05)
ax.set_ylim(-1.5, 1.5)
ax.legend()
save_fig(fig, 'fig1_4_polynomial_data')
plt.show()

## Figures 1.6a–d — Polynomial Fits (M = 0, 1, 3, 9)

In [ ]:
def plot_polyfit(M, name, ax=None, show_title=True):
    """Fit polynomial of degree M and plot."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 4))
    else:
        fig = ax.get_figure()

    coeffs = np.polyfit(x, t, M)
    y_fit = np.polyval(coeffs, x_plot)

    ax.plot(x_plot, np.sin(2 * np.pi * x_plot), color=COLORS['green'], lw=1.5, alpha=0.5, label=r'$\sin(2\pi x)$')
    ax.plot(x_plot, y_fit, color=COLORS['red'], lw=2, label=f'$M = {M}$')
    ax.scatter(x, t, color=COLORS['blue'], s=50, zorder=5, edgecolors='white', linewidths=0.5, label='Data')
    ax.set_xlabel('$x$')
    ax.set_ylabel('$t$')
    ax.set_xlim(-0.05, 1.05)
    ax.set_ylim(-1.5, 1.5)
    if show_title:
        ax.set_title(f'$M = {M}$', fontsize=12)
    ax.legend()
    return fig


# Individual figures
for M, suffix in [(0, 'a'), (1, 'b'), (3, 'c'), (9, 'd')]:
    fig = plot_polyfit(M, f'fig1_6{suffix}_polyfit_M{M}')
    # For M=9, widen y-axis to show wild oscillations
    if M == 9:
        fig.get_axes()[0].set_ylim(-2.5, 2.5)
    save_fig(fig, f'fig1_6{suffix}_polyfit_M{M}')
    plt.show()

## Figure 1.7 — RMSE vs Polynomial Order M

In [ ]:
def compute_rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))


M_values = np.arange(0, 10)
train_rmse = []
test_rmse = []

for M in M_values:
    coeffs = np.polyfit(x, t, M)
    train_rmse.append(compute_rmse(t, np.polyval(coeffs, x)))
    test_rmse.append(compute_rmse(t_test, np.polyval(coeffs, x_test)))

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(M_values, train_rmse, 'o-', color=COLORS['blue'], lw=2, markersize=6, label='Training')
ax.plot(M_values, test_rmse,  'o-', color=COLORS['red'],  lw=2, markersize=6, label='Test')
ax.set_xlabel('$M$')
ax.set_ylabel('$E_{\mathrm{RMS}}$')
ax.set_xticks(M_values)
ax.set_ylim(0, 1.2)
ax.legend()
save_fig(fig, 'fig1_7_rmse_vs_M')
plt.show()

## Figures 1.8a–b — Regularized Polynomial Fits (M = 9)

In [ ]:
def ridge_polyfit(x_data, t_data, M, lam):
    """Ridge regression for polynomial fitting.
    Builds the Vandermonde matrix and solves (X^T X + lambda I) w = X^T t.
    """
    # Build design matrix (Vandermonde)
    X = np.vander(x_data, M + 1, increasing=True)
    I = np.eye(M + 1)
    I[0, 0] = 0  # don't regularize bias
    w = np.linalg.solve(X.T @ X + lam * I, X.T @ t_data)
    return w


def eval_poly(w, x_eval):
    """Evaluate polynomial with coefficients w (increasing order)."""
    X_eval = np.vander(x_eval, len(w), increasing=True)
    return X_eval @ w


def plot_regularized(ln_lam, title_extra, name):
    lam = np.exp(ln_lam)
    w = ridge_polyfit(x, t, 9, lam)
    y_fit = eval_poly(w, x_plot)

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(x_plot, np.sin(2 * np.pi * x_plot), color=COLORS['green'], lw=1.5, alpha=0.5, label=r'$\sin(2\pi x)$')
    ax.plot(x_plot, y_fit, color=COLORS['red'], lw=2, label=f'$M=9,\;\ln\\lambda={ln_lam}$')
    ax.scatter(x, t, color=COLORS['blue'], s=50, zorder=5, edgecolors='white', linewidths=0.5, label='Data')
    ax.set_xlabel('$x$')
    ax.set_ylabel('$t$')
    ax.set_xlim(-0.05, 1.05)
    ax.set_ylim(-1.5, 1.5)
    ax.set_title(f'$M = 9$, $\\ln\\lambda = {ln_lam}$ {title_extra}', fontsize=11)
    ax.legend()
    save_fig(fig, name)
    plt.show()


plot_regularized(-18, '(good regularization)', 'fig1_8a_regularized_good')
plot_regularized(0, '(too much regularization)', 'fig1_8b_regularized_bad')

## Figure 1.10 — RMSE vs ln lambda

In [ ]:
ln_lambdas = np.linspace(-40, 5, 100)
train_rmse_reg = []
test_rmse_reg = []

for ln_lam in ln_lambdas:
    lam = np.exp(ln_lam)
    w = ridge_polyfit(x, t, 9, lam)
    train_rmse_reg.append(compute_rmse(t, eval_poly(w, x)))
    test_rmse_reg.append(compute_rmse(t_test, eval_poly(w, x_test)))

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(ln_lambdas, train_rmse_reg, color=COLORS['blue'], lw=2, label='Training')
ax.plot(ln_lambdas, test_rmse_reg,  color=COLORS['red'],  lw=2, label='Test')
ax.set_xlabel(r'$\ln \lambda$')
ax.set_ylabel('$E_{\mathrm{RMS}}$')
ax.set_ylim(0, 1.2)
ax.legend()
save_fig(fig, 'fig1_10_rmse_vs_lambda')
plt.show()

## Figure 1.15 — MLP Network Diagram

Simple feedforward network: inputs → hidden → outputs.

In [ ]:
def draw_mlp(layer_sizes, layer_labels=None, figsize=(8, 5)):
    """Draw a multi-layer perceptron diagram."""
    fig, ax = plt.subplots(figsize=figsize)
    ax.set_aspect('equal')
    ax.axis('off')

    n_layers = len(layer_sizes)
    max_neurons = max(layer_sizes)
    h_spacing = 2.5
    v_spacing = 1.2
    radius = 0.3

    layer_colors = [COLORS['blue'], COLORS['green'], COLORS['red']]
    if n_layers > 3:
        layer_colors = [COLORS['blue']] + [COLORS['green']] * (n_layers - 2) + [COLORS['red']]

    positions = []
    for i, n in enumerate(layer_sizes):
        layer_pos = []
        x_pos = i * h_spacing
        y_start = (max_neurons - n) * v_spacing / 2
        for j in range(n):
            y_pos = y_start + j * v_spacing
            layer_pos.append((x_pos, y_pos))
        positions.append(layer_pos)

    # Draw connections
    for i in range(n_layers - 1):
        for (x1, y1) in positions[i]:
            for (x2, y2) in positions[i + 1]:
                ax.plot([x1, x2], [y1, y2], color='gray', lw=0.8, alpha=0.5, zorder=1)

    # Draw neurons
    for i, layer_pos in enumerate(positions):
        color = layer_colors[min(i, len(layer_colors) - 1)]
        for (xp, yp) in layer_pos:
            circle = plt.Circle((xp, yp), radius, color=color, ec='white', lw=1.5, zorder=3)
            ax.add_patch(circle)

    # Layer labels
    if layer_labels is None:
        layer_labels = ['Input'] + ['Hidden'] * (n_layers - 2) + ['Output']
    for i, label in enumerate(layer_labels):
        x_pos = i * h_spacing
        y_pos = -1.0
        ax.text(x_pos, y_pos, label, ha='center', va='top', fontsize=11, fontweight='bold',
                color=layer_colors[min(i, len(layer_colors) - 1)])

    ax.set_xlim(-1, (n_layers - 1) * h_spacing + 1)
    ax.set_ylim(-1.8, max_neurons * v_spacing)
    return fig


fig = draw_mlp([3, 5, 5, 2], ['Input\n$D$ features', 'Hidden\nlayer 1', 'Hidden\nlayer 2', 'Output\n$K$ classes'])
save_fig(fig, 'fig1_15_mlp_diagram')
plt.show()

## Figure 1.16 — Compute Growth (petaflop/s-days vs Year)

In [ ]:
milestones = {
    'Perceptron':   (1958, 1e-14),
    'ALVINN':       (1989, 1e-10),
    'LeNet-5':      (1998, 1e-8),
    'RNN Speech':   (2009, 1e-7),
    'AlexNet':      (2012, 1e-2),
    'VGG':          (2014, 1e-1),
    'ResNets':      (2015, 5e-1),
    'NMT':          (2016, 1e2),
    'AlphaGoZero':  (2017, 1e3),
    'GPT-3':        (2020, 3e3),
}

years  = [v[0] for v in milestones.values()]
flops  = [v[1] for v in milestones.values()]
labels = list(milestones.keys())

fig, ax = plt.subplots(figsize=(9, 5))
ax.semilogy(years, flops, 'o-', color=COLORS['blue'], lw=2, markersize=8, zorder=3)

# Label each milestone
offsets = {
    'Perceptron':  (5, 10),
    'ALVINN':      (5, 10),
    'LeNet-5':     (-40, 15),
    'RNN Speech':  (5, -18),
    'AlexNet':     (-55, -5),
    'VGG':         (8, -15),
    'ResNets':     (-55, 10),
    'NMT':         (8, 5),
    'AlphaGoZero': (-80, -10),
    'GPT-3':       (5, -15),
}

for label, yr, fl in zip(labels, years, flops):
    ox, oy = offsets.get(label, (5, 5))
    ax.annotate(label, (yr, fl), textcoords='offset points', xytext=(ox, oy),
                fontsize=8.5, color=COLORS['blue'],
                arrowprops=dict(arrowstyle='->', color='gray', lw=0.7))

ax.set_xlabel('Year')
ax.set_ylabel('Compute (petaflop/s-days)')
ax.set_title('Growth of Training Compute for Notable Models', fontsize=12)
ax.set_xlim(1955, 2023)
save_fig(fig, 'fig1_16_compute_growth')
plt.show()

## Figure 1.9 — Polynomial Coefficients: Overfitting Signature

In [ ]:
# ── Figure 1.9: Coefficient magnitudes with and without regularization ──
M_list = [0, 1, 3, 9]
colors_M = {0: COLORS['blue'], 1: COLORS['green'], 3: COLORS['amber'], 9: COLORS['red']}

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=False)

# ── Left panel: no regularization ──
ax = axes[0]
for M in M_list:
    coeffs = np.polyfit(x, t, M)[::-1]  # increasing order w0..wM
    positions = np.arange(len(coeffs))
    ax.bar(positions + M_list.index(M) * 0.18, np.abs(coeffs), width=0.16,
           color=colors_M[M], label=f'$M={M}$', alpha=0.85)
ax.set_xlabel('Coefficient index $j$')
ax.set_ylabel('$|w_j|$')
ax.set_title('Without regularization', fontsize=11)
ax.set_yscale('log')
ax.set_ylim(1e-2, 1e6)
ax.legend(fontsize=9)

# ── Right panel: with regularization (lambda = exp(-18)) ──
ax = axes[1]
lam = np.exp(-18)
for M in M_list:
    w = ridge_polyfit(x, t, M, lam)  # increasing order
    positions = np.arange(len(w))
    ax.bar(positions + M_list.index(M) * 0.18, np.abs(w), width=0.16,
           color=colors_M[M], label=f'$M={M}$', alpha=0.85)
ax.set_xlabel('Coefficient index $j$')
ax.set_ylabel('$|w_j|$')
ax.set_title(r'With regularization ($\ln\lambda = -18$)', fontsize=11)
ax.set_yscale('log')
ax.set_ylim(1e-2, 1e6)
ax.legend(fontsize=9)

fig.tight_layout()
save_fig(fig, 'fig1_9_coefficients')
plt.show()

## Figure 1.11 — K-Fold Cross-Validation

In [ ]:
# ── Figure 1.11: K-Fold Cross-Validation diagram ──
S = 4  # number of folds
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.axis('off')

bar_height = 0.6
gap = 0.3
seg_width = 2.0

for run in range(S):
    y = (S - 1 - run) * (bar_height + gap)
    ax.text(-0.3, y + bar_height / 2, f'Run {run + 1}', ha='right', va='center',
            fontsize=10, fontweight='bold', color=COLORS['blue'])
    for fold in range(S):
        x0 = fold * seg_width
        if fold == run:
            color = COLORS['red']
            label = 'Test'
        else:
            color = COLORS['blue']
            label = 'Train'
        rect = mpatches.FancyBboxPatch((x0 + 0.05, y), seg_width - 0.1, bar_height,
                                        boxstyle='round,pad=0.05', facecolor=color,
                                        edgecolor='white', linewidth=1.5, alpha=0.85)
        ax.add_patch(rect)
        ax.text(x0 + seg_width / 2, y + bar_height / 2, label,
                ha='center', va='center', fontsize=9, color='white', fontweight='bold')

# Column headers
for fold in range(S):
    x0 = fold * seg_width
    ax.text(x0 + seg_width / 2, S * (bar_height + gap) - gap + 0.25,
            f'Fold {fold + 1}', ha='center', va='bottom', fontsize=10, color=COLORS['blue'])

ax.set_xlim(-1.5, S * seg_width + 0.5)
ax.set_ylim(-0.5, S * (bar_height + gap) + 0.3)
ax.set_title(f'{S}-Fold Cross-Validation', fontsize=12, pad=15)

save_fig(fig, 'fig1_11_cross_validation')
plt.show()

## Figure 1.13 — The Perceptron Model

In [ ]:
# ── Figure 1.13: The Perceptron Model ──
fig, ax = plt.subplots(figsize=(10, 5))
ax.set_aspect('equal')
ax.axis('off')

D = 4  # number of inputs
r_input = 0.35
r_sum = 0.5
r_step = 0.45
r_out = 0.35

# Positions
x_in = 1.0
x_sum = 5.0
x_step = 7.5
x_out = 10.0
y_center = 3.0

input_ys = np.linspace(y_center - (D - 1) * 0.9, y_center + (D - 1) * 0.9, D)
bias_y = input_ys[-1] + 1.5

# Draw input nodes
input_labels = [f'$x_{i+1}$' for i in range(D)]
for i, (iy, lab) in enumerate(zip(input_ys, input_labels)):
    circle = plt.Circle((x_in, iy), r_input, fc=COLORS['blue'], ec='white', lw=1.5, zorder=5)
    ax.add_patch(circle)
    ax.text(x_in, iy, lab, ha='center', va='center', fontsize=10, color='white', fontweight='bold')

# Draw bias node
circle_b = plt.Circle((x_in, bias_y), r_input, fc=COLORS['green'], ec='white', lw=1.5, zorder=5)
ax.add_patch(circle_b)
ax.text(x_in, bias_y, '$1$', ha='center', va='center', fontsize=10, color='white', fontweight='bold')

# Draw summation node
circle_sum = plt.Circle((x_sum, y_center), r_sum, fc=COLORS['amber'], ec='white', lw=2, zorder=5)
ax.add_patch(circle_sum)
ax.text(x_sum, y_center, r'$\Sigma$', ha='center', va='center', fontsize=18, color='white', fontweight='bold')

# Draw step function block
step_box = FancyBboxPatch((x_step - 0.55, y_center - 0.45), 1.1, 0.9,
                           boxstyle='round,pad=0.1', fc=COLORS['purple'], ec='white', lw=1.5, zorder=5)
ax.add_patch(step_box)
ax.text(x_step, y_center, r'$\sigma(\cdot)$', ha='center', va='center', fontsize=11, color='white', fontweight='bold')

# Draw output node
circle_out = plt.Circle((x_out, y_center), r_out, fc=COLORS['red'], ec='white', lw=1.5, zorder=5)
ax.add_patch(circle_out)
ax.text(x_out, y_center, '$y$', ha='center', va='center', fontsize=11, color='white', fontweight='bold')

# Draw weighted connections from inputs to summation
weight_labels = [f'$w_{i+1}$' for i in range(D)]
for i, (iy, wlab) in enumerate(zip(input_ys, weight_labels)):
    ax.annotate('', xy=(x_sum - r_sum - 0.05, y_center + (iy - y_center) * 0.15),
                xytext=(x_in + r_input + 0.05, iy),
                arrowprops=dict(arrowstyle='->', color=COLORS['blue'], lw=1.5))
    # Label weight at midpoint
    mid_x = (x_in + r_input + x_sum - r_sum) / 2
    mid_y = (iy + y_center) / 2
    ax.text(mid_x - 0.15, mid_y + 0.15, wlab, fontsize=9, color=COLORS['blue'],
            ha='center', va='bottom')

# Bias connection
ax.annotate('', xy=(x_sum - r_sum * 0.7, y_center + r_sum * 0.7),
            xytext=(x_in + r_input + 0.05, bias_y),
            arrowprops=dict(arrowstyle='->', color=COLORS['green'], lw=1.5))
ax.text((x_in + r_input + x_sum - r_sum) / 2 - 0.3, (bias_y + y_center) / 2 + 0.35,
        '$b$', fontsize=10, color=COLORS['green'], ha='center', va='bottom')

# Summation to step function
ax.annotate('', xy=(x_step - 0.55, y_center),
            xytext=(x_sum + r_sum + 0.05, y_center),
            arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))

# Step function to output
ax.annotate('', xy=(x_out - r_out - 0.05, y_center),
            xytext=(x_step + 0.55 + 0.05, y_center),
            arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))

# Labels below
ax.text(x_in, input_ys[0] - 1.0, 'Inputs', ha='center', va='top', fontsize=11,
        fontweight='bold', color=COLORS['blue'])
ax.text(x_sum, y_center - r_sum - 0.6, 'Weighted\nSum', ha='center', va='top', fontsize=9,
        fontweight='bold', color=COLORS['amber'])
ax.text(x_step, y_center - 0.45 - 0.6, 'Activation', ha='center', va='top', fontsize=9,
        fontweight='bold', color=COLORS['purple'])
ax.text(x_out, y_center - r_out - 0.6, 'Output', ha='center', va='top', fontsize=11,
        fontweight='bold', color=COLORS['red'])

ax.set_xlim(-0.5, 11.5)
ax.set_ylim(-0.5, bias_y + 1.5)
ax.set_title('The Perceptron Model', fontsize=13, pad=10)

save_fig(fig, 'fig1_13_perceptron')
plt.show()

## Summary

All figures have been saved to the `charts/` directory as both PDF and PNG (300 dpi, transparent background).

In [ ]:
# List all generated chart files
chart_files = sorted([f for f in os.listdir(CHART_DIR) if f.startswith('fig1_')])
print(f'Generated {len(chart_files)} files in {CHART_DIR}:')
for f in chart_files:
    print(f'  {f}')